# Hugging Face Transformers 微调训练入门

本示例将介绍基于 Transformers 实现模型微调训练的主要流程，包括：
- 数据集下载
- 数据预处理
- 训练超参数配置
- 训练评估指标设置
- 训练器基本介绍
- 实战训练
- 模型保存

## YelpReviewFull 数据集

**Hugging Face 数据集：[ YelpReviewFull ](https://huggingface.co/datasets/yelp_review_full)**

### 数据集摘要

Yelp评论数据集包括来自Yelp的评论。它是从Yelp Dataset Challenge 2015数据中提取的。

### 支持的任务和排行榜
文本分类、情感分类：该数据集主要用于文本分类：给定文本，预测情感。

### 语言
这些评论主要以英语编写。

### 数据集结构

#### 数据实例
一个典型的数据点包括文本和相应的标签。

来自YelpReviewFull测试集的示例如下：

```json
{
    'label': 0,
    'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'
}
```

#### 数据字段

- 'text': 评论文本使用双引号（"）转义，任何内部双引号都通过2个双引号（""）转义。换行符使用反斜杠后跟一个 "n" 字符转义，即 "\n"。
- 'label': 对应于评论的分数（介于1和5之间）。

#### 数据拆分

Yelp评论完整星级数据集是通过随机选取每个1到5星评论的130,000个训练样本和10,000个测试样本构建的。总共有650,000个训练样本和50,000个测试样本。

## 下载数据集

In [1]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [3]:
dataset["train"][112]

{'label': 1,
 'text': "I'm not a huge fan of this location. I think that it was oddly built- the small, alley-like front makes it difficult to get past people on your way to/from the bathroom or tables when it's busy. And there's hardly any tables to sit at. Furthermore, people tend to clog the front on their way in, which makes things particularly difficult (especially in the winter weather). The staff were pretty impersonal to be, but maybe that was due to the high traffic of the place and the time I was there. And the coffee that I had was cold- I'm sure it was probably the bottom of the batch. I'd probably only walk in here again if someone else suggested it before or after a movie or while we were shopping in the area."}

In [4]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML

In [5]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [6]:
show_random_elements(dataset["train"])

,label,text
0,4 stars,"My parents were coming to town and my dad has long been a fan of Chuck's Day Off, so this was the perfect opportunity to score a reservation (and score some impress-my-dad points). \n\nIt's not a spot I would ordinarily seek out (I'm more of a diner/dive person than the fine dining type), but I couldn't help but be impressed by the impeccable service (water glasses were never empty, wine recommendations were bang on, and our server took the time to check in on us often).\n\nAppetizers were tasty - a simple cherry tomato salad with fried cheese croutons, and a boston lettuce salad with fried sardines and green goddess dressing were a fantastic start to the evening. \n\nMain courses were short ribs with spaetzle and a scallop dish. Both were perfectly cooked and delicious. \n\nFor dessert we ordered the \""famous\"" deep fried mars bar, and while it was ok, it was nothing to write home about. We shared it between three of us, and couldn't manage more than a few bites each. Major sweetness, though paired with cold vanilla ice cream to offset the liquid sugar factor we made it work. \n\nOverall a fantastic meal. I'd go back again for a splurge - but it's not an everyday dining spot."
1,3 stars,"I\n\nhate\n\nLas Vegas.\n\nThat being said, this was an okay hotel. I got the best room available to the public, and it was $130. And it was decent. So, that's not bad. And the room service buffalo wings were decent, and only $10. And that's not bad. But they wanted to charge me for Internet access. Which is bad. And there was almost no way to plug in my laptop and sit on the bed and put together PowerPoint slides, because the outlet was so far away. And that's not good. And I showed up to check in at 10:00 at night, and it took them 20 minutes to get me checked in. And that's bad. And, you can basically smoke anywhere in this place, much like most of Las Vegas, and that sucks. Listen, I have a lot of bad habits myself, so I'm not judging anyone. But, I try not to inflict my bad habits on others (with fair-to-middling success). 'Nuff said.\n\nAll in all, Flamingo did an average job of housing me for the 22 hours I spent in Vegas. For the benefit of the Las Vegas Tourism Council, I will note:\n\n1. I did not gamble with a single penny of money during that time;\n2. I never left my hotel during that time;\n3. The only alcohol I consumed was free, as part of the seminar I was speaking at.\n\nThus, while my general prohibition on being physically present in Las Vegas was broken (through circumstances beyond my control), I held to my principles as closely as possible."
2,4 stars,"Real cool spot, real cool barber, he's laid back and does a good hair cut. Says pay him what you think is fair. I recommend him to all the white boys out there."
3,4 stars,"Professional, extremely organized and clean. Staff is considerate and helpful."
4,1 star,"Food is always amazing here... and service is great... but today I was in the drive thru and I saw the manager and another emplyee walk out from the trash can area outside... which is fine... but then a few minutes later thet walked back there again... with no trash! Why would they be hiding out in the trash can area? Something doesnt seem right about this. It seems too suspicious. This makes me feel very uncomfortable comming here. I cant say that I know what they were up too, but it didnt look good! Besides... what normal person hangs out with another dude behind the stinky trash cans? \n\nThis manager was wearing a blue shirt and had long dark hair in a pony tail. The other employee was wearing a red shirt and a hat. Looked like he had a shaved head. Both were skinny. \n\nNot a good look for you guys!!"
5,2 star,"Went this past weekend with my husband. Drove all the way from Chandler actually to check this place out. When we walked in we were expecting some sort of greet. The bartender was too busy to say hello and it wasn't until another few minutes the server comes up to us and tells 

## 预处理数据

下载数据集到本地后，使用 Tokenizer 来处理文本，对于长度不等的输入数据，可以使用填充（padding）和截断（truncation）策略来处理。

Datasets 的 `map` 方法，支持一次性在整个数据集上应用预处理函数。

下面使用填充到最大长度的策略，处理整个数据集：

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/gl/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
show_random_elements(tokenized_datasets["train"], num_examples=1)

,label,text,input_ids,token_type_ids,attention_mask
0,2 star,"The first time i walked into One Love Vapors was back in early October and i knew they were just newly opened. Saw them on yelp and decided to see whats up. I walked in and everything was nice, some music playing, i was greeted, and the guy helping me was very energetic and very friendly. \n\nNow, I haven't been able to drive by again and visit until just about a week ago. I wanted to see what was new, their juices, see if the same guys work there, and just to hang out. I walked in, it was extremely quiet, the guy who i remember came out and asked what i needed. I just wanted to try some juice and he gave me the menu(s). i chose a few and he let me try them. out of maybe 5 flavors, 3 tasted burnt. unfortunately. This guy changed in a few months i think. Either he was just tired that day, or he just doesnt like the business. This visit was nothing like my first. Customer Service completely fell and i'm uninterested in ever wanting to visit.\n\n2 stars for a couple juices that i sampled and liked.","[101, 1109, 1148, 1159, 178, 2045, 1154, 1448, 2185, 159, 11478, 3864, 1108, 1171, 1107, 1346, 1357, 1105, 178, 1450, 1152, 1127, 1198, 3599, 1533, 119, 20318, 1172, 1113, 6798, 1233, 1643, 1105, 1879, 1106, 1267, 1184, 1116, 1146, 119, 146, 2045, 1107, 1105, 1917, 1108, 3505, 117, 1199, 1390, 1773, 117, 178, 1108, 11196, 117, 1105, 1103, 2564, 4395, 1143, 1108, 1304, 20069, 1105, 1304, 4931, 119, 165, 183, 165, 183, 2249, 4064, 117, 146, 3983, 112, 189, 1151, 1682, 1106, 2797, 1118, 1254, 1105, 3143, 1235, 1198, 1164, 170, 1989, 2403, 119, 146, 1458, 1106, 1267, 1184, 1108, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...]"


### 数据抽样

使用 1000 个数据样本，在 BERT 上演示小规模训练（基于 Pytorch Trainer）

`shuffle()`函数会随机重新排列列的值。如果您希望对用于洗牌数据集的算法有更多控制，可以在此函数中指定generator参数来使用不同的numpy.random.Generator。

In [9]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(650000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(10000))

## 微调训练配置

### 加载 BERT 模型

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 训练超参数（TrainingArguments）

完整配置参数与默认值：https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/trainer#transformers.TrainingArguments

源代码定义：https://github.com/huggingface/transformers/blob/v4.36.1/src/transformers/training_args.py#L161

**最重要配置：模型权重保存路径(output_dir)**

In [11]:
from transformers import TrainingArguments

model_dir = "models/bert-base-cased-finetune-yelp"

# logging_steps 默认值为500，根据我们的训练数据和步长，将其设置为100
training_args = TrainingArguments(output_dir=model_dir,
                                  per_device_train_batch_size=16,
                                  num_train_epochs=5,
                                  logging_steps=100)

In [12]:
# 完整的超参数配置
print(training_args)

TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=no,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
gradient_checkpointing_kwargs=None,
greater_is_better=None,
group_by_le

### 训练过程中的指标评估（Evaluate)

**[Hugging Face Evaluate 库](https://huggingface.co/docs/evaluate/index)** 支持使用一行代码，获得数十种不同领域（自然语言处理、计算机视觉、强化学习等）的评估方法。 当前支持 **完整评估指标：https://huggingface.co/evaluate-metric**

训练器（Trainer）在训练过程中不会自动评估模型性能。因此，我们需要向训练器传递一个函数来计算和报告指标。 

Evaluate库提供了一个简单的准确率函数，您可以使用`evaluate.load`函数加载

In [13]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")


接着，调用 `compute` 函数来计算预测的准确率。

在将预测传递给 compute 函数之前，我们需要将 logits 转换为预测值（**所有Transformers 模型都返回 logits**）。

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### 训练过程指标监控

通常，为了监控训练过程中的评估指标变化，我们可以在`TrainingArguments`指定`evaluation_strategy`参数，以便在 epoch 结束时报告评估指标。

In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir=model_dir,
                                  evaluation_strategy="epoch", 
                                  per_device_train_batch_size=32,
                                  num_train_epochs=3,
                                  logging_steps=30)

## 开始训练

### 实例化训练器（Trainer）

`kernel version` 版本问题：暂不影响本示例代码运行

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

## 使用 nvidia-smi 查看 GPU 使用

为了实时查看GPU使用情况，可以使用 `watch` 指令实现轮询：`watch -n 1 nvidia-smi`:

```shell
Every 1.0s: nvidia-smi                                                   Wed Dec 20 14:37:41 2023

Wed Dec 20 14:37:41 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   64C    P0              69W /  70W |   6665MiB / 15360MiB |     98%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A     18395      C   /root/miniconda3/bin/python                6660MiB |
+---------------------------------------------------------------------------------------+
```

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.783000,0.730225,0.684300
2,0.639400,0.708611,0.689900
3,0.561800,0.737835,0.690400


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



TrainOutput(global_step=60939, training_loss=0.6712397507678011, metrics={'train_runtime': 89674.1195, 'train_samples_per_second': 21.745, 'train_steps_per_second': 0.68, 'total_flos': 5.130803778048e+17, 'train_loss': 0.6712397507678011, 'epoch': 3.0})

In [18]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(100))

In [19]:
trainer.evaluate(small_test_dataset)

{'eval_loss': 0.8097428679466248,
 'eval_accuracy': 0.65,
 'eval_runtime': 1.6845,
 'eval_samples_per_second': 59.363,
 'eval_steps_per_second': 7.717,
 'epoch': 3.0}

### 保存模型和训练状态

- 使用 `trainer.save_model` 方法保存模型，后续可以通过 from_pretrained() 方法重新加载
- 使用 `trainer.save_state` 方法保存训练状态

In [23]:
trainer.save_model(model_dir)

In [24]:
trainer.save_state()

In [25]:
 trainer.model.save_pretrained("./")

## Homework: 使用完整的 YelpReviewFull 数据集训练，看 Acc 最高能到多少